# TP 2 – Top 10 des zones avec les RDD (30 min)

**Objectif** : trouver les 10 zones de départ (`PULocationID`) qui rapportent le plus (`total_amount`), avec l'API RDD.

Colonnes de `trips.csv` (index à partir de 0) : `PULocationID` = colonne **7**, `total_amount` = colonne **16**.

Livrable : ce notebook, enregistré dans le dépôt (`git checkout -b prenom` puis `git add` / `git commit`).

**Réussi si** : le top 10 affiche les noms des zones, JFK Airport en tête (≈ 261 823 $).

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("TP")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "—", spark.sparkContext.uiWebUrl)

### 1. Charger `data/trips.csv` en RDD de lignes (`sc.textFile`) et retirer l'en-tête

Indice : `first()` renvoie la première ligne ; `filter(lambda l: l != entete)`.

In [ ]:
lignes = ...
entete = ...
courses = ...
print(courses.count())

### 2. Construire un PairRDD `(PULocationID, total_amount)`

Indice : `map(lambda l: l.split(","))` puis `map(lambda c: (int(c[7]), float(c[16])))`.

In [ ]:
paires = ...
paires.take(3)

### 3. Chiffre d'affaires par zone avec `reduceByKey`

In [ ]:
ca_par_zone = ...

### 4. Garder le top 10 avec `takeOrdered`

Indice : `takeOrdered(10, key=lambda kv: -kv[1])`.

In [ ]:
top10 = ...
top10

### 5. Afficher le nom des zones : `join` avec `data/taxi_zone_lookup.csv`

Le fichier met tous les champs entre guillemets : préférez `csv.reader` à un `split(",")`.

```python
import csv
champs = next(csv.reader([ligne]))   # ['132', 'Queens', 'JFK Airport', 'Airports']
```
Indices :
- comme pour `trips.csv`, **retirez l'en-tête** du fichier des zones avant de convertir ;
- ⚠ les clés doivent avoir **le même type des deux côtés** : `int(c[0])`. Avec des clés `'132'` (texte) d'un côté et `132` (entier) de l'autre, le `join` renvoie un résultat **vide, sans aucune erreur** ;
- construire un PairRDD `(LocationID, Zone)`, puis `sc.parallelize(top10).join(zones)` ;
- le `join` ne garde pas l'ordre : retriez avec `sortBy(lambda kv: -kv[1][0])`.

In [ ]:
import csv
zones = ...
resultat = ...
for zid, (ca, nom) in resultat.collect():
    print(nom, round(ca, 2))

### 6. Dans la Spark UI : combien de **stages** pour le calcul du top 10 ? Où est le **shuffle** ?

Écrivez votre réponse ici :

_Votre réponse…_

### Bonus – `groupByKey` contre `reduceByKey`

Refaites l'étape 3 avec `groupByKey().mapValues(sum)` et lancez un `count()` sur chaque version.
Dans la Spark UI, onglet **Stages**, comparez la colonne **Shuffle Write** des deux stages. Combien de fois plus de données `groupByKey` envoie-t-il sur le réseau ? Pourquoi ?

(Ne comparez pas les temps : sur 120 000 lignes et un seul portable, la différence est noyée dans le bruit. Sur un vrai cluster, c'est le volume du shuffle qui coûte.)

In [ ]:
...